Step 2

In [1]:
"""
Espoo District Heating Network Optimization
"""


import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import dhnx
import os


import os

# Adjust this path to match your actual project structure
base_dir = "/Users/matis/Desktop/KTH/Practical Optimisation/Tutorial 1/Network_optmisation/Project__DH"
twn_data_path = os.path.join(base_dir, "DHNx_files", "Espoo", "twn_data_step2")
invest_data_path = os.path.join(base_dir, "DHNx_files", "Espoo", "invest_data_step2")


print('='*60)
print('ESPOO DHN - INVESTMENT OPTIMIZATION')
print('='*60)

# Load network
print('\n[1/3] Loading network...')
network = dhnx.network.ThermalNetwork()
network = network.from_csv_folder(twn_data_path)
invest_opt = dhnx.input_output.load_invest_options(invest_data_path)

print(f'  - Producers: {len(network.components.producers)}')
print(f'  - Consumers: {len(network.components.consumers)}')
print(f'  - Pipe segments: {len(network.components.pipes)}')

# Plot initial network topology
print('\nPlotting initial network...')
plt.figure(figsize=(10, 8))
static_map_initial = dhnx.plotting.StaticMap(network)
static_map_initial.draw(background_map=False)
plt.scatter(network.components.producers['lon'], network.components.producers['lat'],
            color='tab:red', label='producers', zorder=2.5, s=150)  # Changed to tab:red
plt.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
            color='tab:green', label='consumers', zorder=2.5, s=100)  # Changed to tab:green
plt.scatter(network.components.forks['lon'], network.components.forks['lat'],
            color='tab:grey', label='forks', zorder=2.5, s=50)  # Changed to tab:grey
plt.title('Espoo DHN - Initial Network Topology', fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Outputs/network_initial_step2.png', dpi=150, bbox_inches='tight')
plt.close()


# Run optimization
print('\n[2/3] Running optimization...')
network.optimize_investment(invest_options=invest_opt, solver='glpk')


# Get results
results = network.results.optimization['components']['pipes']
results.to_csv("Outputs/optimization_results.csv")

# Summary
print('\n[3/3] Results:')
print('-'*60)
objective = network.results.optimization['oemof_meta']['objective']
print(f'Total Cost: {objective:,.0f} EUR')

# Pipe types used
active_pipes = results[results['capacity'] > 0.001]
pipe_counts = active_pipes['hp_type'].value_counts()
print(f'\nPipes installed:')
for pipe_type, count in pipe_counts.items():
    total_cap = active_pipes[active_pipes['hp_type'] == pipe_type]['capacity'].sum()
    print(f'  {pipe_type}: {count} segments ({total_cap:.1f} kW)')

# Plot optimized network (Following teacher's approach)
print('\nCreating optimized network plot...')
twn_results = network
twn_results.components['pipes'] = results[results['capacity'] > 0.001]

plt.figure(figsize=(10, 8))
static_map_2 = dhnx.plotting.StaticMap(twn_results)
static_map_2.draw(background_map=False)
plt.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
            color='tab:green', label='consumers', zorder=2.5, s=100)
plt.scatter(network.components.producers['lon'], network.components.producers['lat'],
            color='tab:red', label='producers', zorder=2.5, s=150)
plt.scatter(network.components.forks['lon'], network.components.forks['lat'],
            color='tab:grey', label='forks', zorder=2.5, s=50)
plt.title('Espoo DHN - Optimized Network', fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Outputs/network_optimized_step2.png', dpi=150, bbox_inches='tight')
plt.close()


print('\nGenerated files:')
print('  - Outputs/network_initial.png')
print('  - Outputs/network_optimized.png')
print('  - Outputs/optimization_results.csv')

Need to install CoolProp to use the hydraulic pre-calculation module.
ESPOO DHN - INVESTMENT OPTIMIZATION

[1/3] Loading network...
  - Producers: 5
  - Consumers: 10
  - Pipe segments: 20

Plotting initial network...

[2/3] Running optimization...


d:\Anaconda\envs\env_P2\lib\site-packages\oemof\solph\flows\_flow.py:163: FutureWarning: For backward compatibility, the option investment overwrites the option nominal_value. Both options cannot be set at the same time.
  warn(msg, FutureWarning)
d:\Anaconda\envs\env_P2\lib\site-packages\oemof\network\network\nodes.py:250: FutureWarning: Usage of oemof.network.Component is deprecated. Use oemof.network.Node instead.
  warnings.warn(


GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 --write C:\Users\matis\AppData\Local\Temp\tmp1pj29y_o.glpk.raw --wglp C:\Users\matis\AppData\Local\Temp\tmp_15yap29.glpk.glp
 --cpxlp C:\Users\matis\AppData\Local\Temp\tmpsxfiggnc.pyomo.lp
Reading problem data from 'C:\Users\matis\AppData\Local\Temp\tmpsxfiggnc.pyomo.lp'...
C:\Users\matis\AppData\Local\Temp\tmpsxfiggnc.pyomo.lp:7308: warning: lower bound of variable 'x127' redefined
C:\Users\matis\AppData\Local\Temp\tmpsxfiggnc.pyomo.lp:7308: warning: upper bound of variable 'x127' redefined
1146 rows, 1005 columns, 2605 non-zeros
125 integer variables, all of which are binary
7433 lines were read
Writing problem data to 'C:\Users\matis\AppData\Local\Temp\tmp_15yap29.glpk.glp'...
6399 lines were written
GLPK Integer Optimizer, v4.65
1146 rows, 1005 columns, 2605 non-zeros
125 integer variables, all of which are binary
Preprocessing...
1116 rows, 1000 columns, 2550 non-zeros
125 integer variables, all of whic

INFO:root:Optimization successful...



[3/3] Results:
------------------------------------------------------------
Total Cost: 19,892,904 EUR

Pipes installed:
  DN30: 14 segments (7688.1 kW)
  DN50: 2 segments (7931.7 kW)

Creating optimized network plot...

Generated files:
  - Outputs/network_initial.png
  - Outputs/network_optimized.png
  - Outputs/optimization_results.csv
